# 🚀 Evaluate the distilled shopping agent on SageMaker AI: ShoppingBench ASR and IFEval

Runs these models as agents on the same held-out ShoppingBench problems:

<table align="left">
  <tr><th align="left"><b>Model</b></th><th align="left"><b>What it is</b></th><th align="left"><b>Served from</b></th></tr>
  <tr><td align="left">Base</td><td align="left">Qwen3.5-4B before distillation</td><td align="left">SageMaker AI real-time endpoint</td></tr>
  <tr><td align="left">Student</td><td align="left">The distilled model (Model Package from the training notebook)</td><td align="left">SageMaker AI real-time endpoint</td></tr>
  <tr><td align="left">frontier_medium (optional)</td><td align="left">A strong frontier model, as a comparison point</td><td align="left">Cached reference results</td></tr>
  <tr><td align="left">reference_qwen3-32b (optional)</td><td align="left">Qwen3 32B dense — same model family as base/student, much larger</td><td align="left">Cached reference results</td></tr>
</table>
<br clear="all">

Reported per model: **ASR** (Absolute Success Rate of solving user shopping queries) and, optionally, **IFEval**
(General instruction following).

## About ShoppingBench

[ShoppingBench](https://github.com/yjwjy/ShoppingBench) ([paper](https://arxiv.org/abs/2508.04266)) gives an agent a shopper's request and tools over "a large-scale shopping sandbox incorporating over 2.5 million real-world products". The agent must search, inspect products, and recommend the one(s) that meet every requirement.

We evaluate three **intents** of increasing difficulty (`EVAL_TASKS` in the configuration):

<table align="left">
  <tr><th align="left"><b>Intent</b></th><th align="left"><b>The shopper wants</b></th><th align="left"><b>Solved when</b></th></tr>
  <tr><td align="left"><code>product</code></td><td align="left">One product meeting a set of constraints</td><td align="left">The recommended product meets every requirement</td></tr>
  <tr><td align="left"><code>shop</code></td><td align="left">Several products, all from one shop</td><td align="left">Every product meets every requirement, and all come from one shop</td></tr>
  <tr><td align="left"><code>voucher</code></td><td align="left">Several products within a budget after a voucher</td><td align="left">Every product meets every requirement, and the total after the voucher fits the budget</td></tr>
</table>
<br clear="all">

Example requests:
- **product:** "Women's jeans in size eu:30, baggy fit, high waist, denim, plain, above 114 PHP."
- **shop:** "Find shops offering gold jewelry that include: a necklace with zircon gemstones, a gold ring ..."
- **voucher:** "...a phone strap, a fuel pump filter, and an HP black ink bottle. My budget is only 398, but I have a voucher ..."

## Environment (once, then switch kernel)

Installs the notebook's packages into a venv registered as the **seqkd-eval** kernel (the harness and lm-eval get their
own venvs later). The harness and scoring run on this machine (CPU; e.g. a Studio `ml.m5.4xlarge` space or a laptop);
only the models run on SageMaker. Run the next cell, switch the kernel to **seqkd-eval**, and continue from 0b.

In [ ]:
import subprocess, sys
from pathlib import Path

WORK = Path.home() / "shopbench-eval"
WORK.mkdir(exist_ok=True)
EVAL_ENV = WORK / "evalenv"
PY = EVAL_ENV / "bin" / "python"
PIP = EVAL_ENV / "bin" / "pip"

PACKAGES = [   # pinned to the versions resolved at authoring time for reproducibility
    "sagemaker==3.23.0",
    "openai==3.26.0",
    "huggingface_hub==1.33.0",
    "transformers==5.19.0",
    "pandas==3.0.6",
    "matplotlib==3.11.2",
    "httpx==0.28.1",
    "requests==2.34.2",
    "boto3==1.43.108",
    "ipykernel==7.4.0",
    "ipywidgets==8.1.9",   # lets rich (sagemaker-core's console output) render live displays as Jupyter widgets
]

# Create the venv (once), install the packages, and register it as the 'seqkd-eval' Jupyter kernel.
if not PY.exists():
    subprocess.run([sys.executable, "-m", "venv", str(EVAL_ENV)], check=True)
subprocess.run([str(PIP), "install", "-q", "-U", "pip", *PACKAGES], check=True)
subprocess.run([str(PY), "-m", "ipykernel", "install", "--user",
                "--name", "seqkd-eval", "--display-name", "seqkd-eval"], check=True)

print("Kernel 'seqkd-eval' registered. Switch to it (Kernel > Change Kernel), then run the cells below.")

## Configuration

The next cell sets everything the run needs. The knobs you'll most likely touch:

- **`BASE_HF`** — the base model (`Qwen/Qwen3.5-4B`); the student is its distilled LoRA adapter.
- **`MODEL_PKG_ARN`** — the Model Package produced by the training notebook (loaded automatically from `%store` in 0b-prev; set it by hand if needed).
- **`EVAL_TASKS`** — which intents to run (`product`, `shop`, `voucher`).
- **`MAX_PROBLEMS`** — set to a small int (e.g. 5) for a quick end-to-end smoke test before the full run.
- **`CACHED_REF_TAGS`** — the reference/frontier comparison rows, served entirely from frozen cached results under `out/reference_evaluations/` (never re-invoked). Add/remove a tag to show/hide a comparison row; its rollout files must exist in the archive.
- **`REUSE_ENDPOINTS`** — `{}` to deploy fresh, or `{tag: (endpoint, inference_component)}` to reuse an already-running endpoint and skip redeploy.
- **`INSTANCE_TYPES`** — the ordered GPU preference list for the endpoint (first = preferred).

**IAM.** The execution role needs: create/invoke SageMaker models, endpoints, and inference components; and read access to the Model Package's S3 bucket (without it the endpoint fails with only "Request to service failed").

In [ ]:
#retrieve MODEL_PKG_ARN from previous notebook
%store -r MODEL_PKG_ARN

if MODEL_PKG_ARN: 
    print(f"Model Package ARN: {MODEL_PKG_ARN}")

In [ ]:
import os, json, sys, time, subprocess, logging
from pathlib import Path
assert "evalenv" in sys.executable, f"switch the kernel to 'seqkd-eval' first (current: {sys.executable})"
import boto3

REGION = boto3.Session().region_name or "us-west-2"
from sagemaker.core.helper.session_helper import get_execution_role
try:
    ROLE = get_execution_role()
except Exception:
    ROLE = "<arn:aws:iam::ACCOUNT:role/your-sagemaker-execution-role>"   # running outside Studio: set it explicitly

# ---- what to evaluate ----
BASE_HF = "Qwen/Qwen3.5-4B"

# Reference/frontier comparison rows, served from a frozen local cache (out/reference_evaluations/), never
# re-invoked. The restore cell (step 4) copies each tag's cached rollout+cfg into OUT; run_rollouts() then
# sees them as complete and skips them, so no live call (or model ID) is needed — just the tag.
#   frontier_medium      — a strong frontier model (comparison point)
#   reference_qwen3-32b  — Qwen3 32B dense, same family as base/student
CACHED_REF_TAGS = ["reference_qwen3-32b", "frontier_medium"]
ENABLE_THINKING = False
EVAL_TASKS = ["product", "shop", "voucher"]   # held-out problems: 103 / 105 / 116
TRAIN_CORPUS = "oro-ai/sn15-shoppingbench-sft-15k"
MAX_PROBLEMS = None                           # e.g. 5 for a quick end-to-end test
RUN_IFEVAL = True

# ---- Endpoints: base + student LoRA adapter as inference components on ONE DJL-LMI endpoint (step 3) ----
# Reuse already-InService resources: {tag: (endpoint_name, inference_component_name)}; {} = deploy fresh.
# Only "base" is read — the deploy cell always (re)attaches the student adapter idempotently.
REUSE_ENDPOINTS = {}
MAX_MODEL_LEN = 40960
# Ordered instance preference; SageMaker falls back down the list on capacity shortage. Under an FTP
# (TRAINING_PLAN_ARN below) this MUST be a single type matching the plan.
INSTANCE_TYPES = ["ml.g5.2xlarge","ml.g6.2xlarge","ml.g6e.2xlarge"]

# ---- Flexible Training Plan (FTP), optional ----
# A training-plan ARN deploys onto reserved capacity instead of on-demand; None = on-demand. A reservation
# covers one instance type, so when set, INSTANCE_TYPES must be a single matching entry.
TRAINING_PLAN_ARN = None                      # e.g. "arn:aws:sagemaker:us-west-2:ACCOUNT:training-plan/NAME"

# Fail fast on the FTP/multi-type mismatch, before any deploy work (kd_serve enforces the same rule at deploy).
assert not (TRAINING_PLAN_ARN and len(INSTANCE_TYPES) != 1), (
    f"A Flexible Training Plan reservation covers a single instance type, so set INSTANCE_TYPES to exactly "
    f"one type matching the plan when TRAINING_PLAN_ARN is set. Got {len(INSTANCE_TYPES)}: {INSTANCE_TYPES}")

# ---- Agent harness ----
# ROLLOUT_THREADS = concurrent trajectories per job; total search demand ~= ROLLOUT_THREADS * n_jobs. Lower it
# if search timeouts / queue-depth warnings appear.
ROLLOUT_THREADS = 4
WORK = Path.home() / "shopbench-eval"   # harness, index, venvs and outputs (~15 GB)
RUN_NAME = ""                           # use a new name per run; finished trajectories in OUT are reused
OUT = WORK / "out" / RUN_NAME if RUN_NAME else WORK / "out"
OUT.mkdir(parents=True, exist_ok=True)
RELAY_PORT = 8101                       # first local relay port

SCRIPTS = Path("scripts").resolve()
assert (SCRIPTS / "shopbench_harness.py").exists(), "run the notebook from its own folder (distillation/)"
sys.path.insert(0, str(SCRIPTS))
from sm_utils import run       # stream a subprocess's output live into the notebook; raises on non-zero exit
for name in ("botocore", "httpx", "httpx2"):
    logging.getLogger(name).setLevel(logging.WARNING)

print(f"Region: {REGION}\nRole:{ROLE}\nModel Package ARN: {MODEL_PKG_ARN}")

### Preflight: can the account host the endpoint?

Before deploying, confirm there's enough **endpoint-usage** quota — a capacity failure part-way through a deploy wastes time and leaves resources to clean up. Base + student share **one** endpoint: the base inference component owns the single instance (provisioned over the `INSTANCE_TYPES` preference list), and the student LoRA adapter rides on it with no compute of its own. So the run needs just **1 instance total** (0 if an endpoint is reused).

The check reads the per-type `"<type> for endpoint usage"` Service Quotas (via `scripts/quota_checker.py`, cached on disk) and passes if any type in the preference list has room. It raises on an outright FAIL so you don't start a deploy that can't succeed.

In [ ]:
from quota_checker import EndpointQuotaPreflight

N_NEW = 0 if REUSE_ENDPOINTS else 1          # one base-IC instance; the adapter IC borrows the base's compute
print(f"reusing {list(REUSE_ENDPOINTS) or 'none'}\n")
EndpointQuotaPreflight(region=REGION).check(
    instance_types=INSTANCE_TYPES, n_new_endpoints=N_NEW)

## 1. Product search: Lucene index and search server

The tools query a **local copy of the catalog** through a search server, so every model sees the same results.

**The catalog** (`resources/documents.jsonl.gz`, 1.4 GB) has one JSON record per product, 2,746,368 in total:

```json
{"id": "126564",
 "contents": "Rainbow Loom Refill (Navy Blue Opaque)\nnavy blue 5 10\nloom bands 6 years and above ...",
 "product": {"product_id": "126564", "shop_id": "7889", "title": "Rainbow Loom Refill (Navy Blue Opaque)",
             "price": 75.0, "sold_count": 1, "service": ["official"],
             "sku_options": {"1": {"color family": "navy blue", ...}},
             "attributes": {"recommended_age": ["6 years and above"], ...}, ...}}
```

- `contents`: the searchable text (title, SKU option values, attribute values).
- `product`: the full catalog record. The tools return fields from it, and the scorer checks it for recommended IDs.

**The index.** [Pyserini](https://github.com/castorini/pyserini) builds a [Lucene](https://lucene.apache.org/) inverted index of `contents` (Java, hence the JDK): each word, lower-cased and stemmed, maps to the products containing it. 3.0 GB, built once.

**The server.** A Flask app on `127.0.0.1:5631` with one route per tool.

`find_product`:
1. Ranks all products against `q` with **BM25** keyword relevance.
2. Drops products failing the optional filters (`shop_id`, `price`, `service`) and keeps the first 50.
3. Optionally re-sorts those 50 by sold count or price.
4. Returns page `page` (1–5) of 10.

`view_product_information` looks up each ID directly and returns descriptions, SKU options and attributes.

What this means for the agent:
- Search is **keyword** matching, not semantic: wording the query like product titles matters.
- Sorting only reorders the 50 most relevant matches (`priceasc` = cheapest *relevant* product).
- Size, color and other variant details are only visible through `view_product_information`.

If many tool calls fail (`null_tool_results` in the health checks), restart the server with `harness.start_search()`.

In [ ]:
run("bash", SCRIPTS / "setup_shopbench.sh", WORK, env={**os.environ, "PYTHON": sys.executable})   # skips finished steps

from shopbench_harness import Harness, nlines
harness = Harness(WORK)
SB_DIR = harness.sb                     # the ShoppingBench checkout
if not harness.search_ok(): harness.start_search(workers=12)   # 12 gunicorn workers to use the idle cores
print("search server up")

## 2. Decontaminated test problems

A fair evaluation has to measure *generalization*, not memorization. The student was distilled on `TRAIN_CORPUS`, and **147 of the 250 `product` test queries appear verbatim in that training data** — scoring on those would reward the student for recalling problems it was trained on and inflate its apparent ASR.

So we **decontaminate**: only problems whose query never appears in `TRAIN_CORPUS` are kept (the held-out set). This is why the counts printed below are smaller than the full test set — and why the base model is evaluated on the *same* held-out problems, so the base→student comparison is apples-to-apples.

`scripts/make_clean_split.py` finds the held-out problems and writes them to `data/synthesize_<task>_test_clean.jsonl`.

In [ ]:
run(sys.executable, SCRIPTS / "make_clean_split.py", "--shopbench", SB_DIR, "--corpus", TRAIN_CORPUS, "--tasks", *EVAL_TASKS)
CLEAN = {t: SB_DIR / "data" / f"synthesize_{t}_test_clean.jsonl" for t in EVAL_TASKS}   # held-out problems
if MAX_PROBLEMS:                        # quick test: only the first N held-out problems of each intent
    for t in EVAL_TASKS:
        rows = open(CLEAN[t]).readlines()[:MAX_PROBLEMS]
        CLEAN[t] = CLEAN[t].with_name(f"synthesize_{t}_test_clean_first{MAX_PROBLEMS}.jsonl")
        CLEAN[t].write_text("".join(rows))
print({t: sum(1 for _ in open(p)) for t, p in CLEAN.items()})

## 3. Deploy the base and student on one endpoint (inference components)

Both models share **one** real-time endpoint, as SageMaker **inference components (ICs)**:

- **Base** (`base` IC): downloads `Qwen/Qwen3.5-4B` from Hugging Face at container start, with LoRA enabled. It owns the instance + GPU.
- **Student** (`student` adapter IC): the distilled **LoRA adapter** from the Model Package (`checkpoints/hf/`), mounted on the base IC — it borrows the base's compute, so the base weights load **once** and the adapter is applied per request.

This needs **one instance total**. The endpoint's instance preference list (`INSTANCE_TYPES`) lets SageMaker provision from the highest-priority type with capacity and fall back across the rest. At invoke time the model is selected by `InferenceComponentName` (base vs student). Served by the DJL-LMI (vLLM) container via `scripts/kd_serve.py`. Endpoints bill while in service; step 9 deletes everything.

**Reserved capacity (optional).** To guarantee GPU availability — e.g. for a scheduled eval when on-demand `g6e`/`g6` is scarce — set `TRAINING_PLAN_ARN` in the config cell to a **Flexible Training Plan (FTP)** reservation (target resource `endpoint`). The endpoint then runs on that reserved capacity with `capacity-reservations-only`, so it never falls back to on-demand and only serves during the reservation window. A reservation covers **exactly one instance type**, so FTP is incompatible with the multi-type fallback pool: when a plan ARN is set, `INSTANCE_TYPES` must contain a single type that matches the plan's type (`kd_serve` raises a `ValueError` otherwise). Leave `TRAINING_PLAN_ARN = None` for ordinary on-demand deployment with the full fallback pool.

In [ ]:
import kd_serve
from sagemaker.core import image_uris

# Deploy base + student as inference components (see the design section above). base vs student is selected
# at invoke time by InferenceComponentName (the relay passes it; the OpenAI route can't address an IC).
STUDENT_IC = "student"          # suffix for the adapter IC on the endpoint
ENDPOINT_NAME = f"seqkd-eval-{time.strftime('%Y%m%d-%H%M%S')}"
# Serving container is owned here, not in kd_serve. Pin the LMI version for reproducible redeploys.
IMAGE_URI = image_uris.retrieve("djl-lmi", region=REGION, version="0.36.0")

if TRAINING_PLAN_ARN:
    print(f"FTP: deploying on reserved capacity {TRAINING_PLAN_ARN} ({INSTANCE_TYPES})")

# Base: reuse an InService base IC if provided, else deploy one.
if "base" in REUSE_ENDPOINTS:
    ep, base_ic = REUSE_ENDPOINTS["base"]
    status = kd_serve.ic_status(base_ic, region=REGION)
    assert status == "InService", (
        f"reused base IC {base_ic!r} is {status or 'missing (torn down)'}, not InService — "
        f"set REUSE_ENDPOINTS={{}} in the config cell to deploy fresh.")
    print(f"reusing base IC {base_ic} on {ep} (InService)")
else:
    ep, base_ic = kd_serve.deploy_base_endpoint(
        endpoint_name=ENDPOINT_NAME, role=ROLE, region=REGION, image_uri=IMAGE_URI,
        base_model_id=BASE_HF, instance_types=INSTANCE_TYPES, training_plan_arn=TRAINING_PLAN_ARN,
        max_lora_rank=64, max_loras=4, max_model_len=MAX_MODEL_LEN)
    print(f"deployed base IC {base_ic} on {ep}")

# Student: resolve the adapter artifacts and idempotently attach them as an adapter IC on the base
# (reused if the artifact matches, redeployed if it changed). The base weights are already loaded and shared.
adapter_uri = kd_serve.resolve_adapter_uri(MODEL_PKG_ARN, REGION)
print("student adapter:", adapter_uri)
adapter_ic = kd_serve.attach_adapter(
    endpoint_name=ep, base_ic_name=base_ic,
    adapter_component_name=f"{ep}-{STUDENT_IC}", adapter_s3_uri=adapter_uri, region=REGION)
print(f"student adapter IC {adapter_ic} attached")

ENDPOINTS = {"base": (ep, base_ic), "student": (ep, adapter_ic)}
print(ENDPOINTS)

## 4. Connect the harness to the endpoints

The harness uses a plain OpenAI client. A small local relay per model forwards each request to the SageMaker endpoint via `invoke_endpoint` (SigV4, so multi-hour runs never fail on token expiry), routing to the right inference component by name — base vs the student adapter — since the OpenAI passthrough route can't select an IC. The reference/frontier comparison models aren't called live here: they're served from frozen cached rollouts, so they have no relay. The smoke test checks the thinking switch reaches base and student: `reasoned` should be `True` with thinking on and `False` with it off.

In [ ]:
from openai import OpenAI
import gc, http.server
import sm_openai_relay as relay

# Free ports held by relays from a previous run (avoids "address already in use"): stop_all() clears this
# run's relays; the gc sweep also catches any left by an earlier run.
relay.stop_all()
for _srv in [o for o in gc.get_objects() if isinstance(o, http.server.HTTPServer)]:
    try:
        _srv.shutdown(); _srv.server_close()
    except Exception:
        pass

# One relay per LIVE model (base + student share one endpoint, differ only by IC, so each relay pins its IC).
URLS = {tag: relay.start_relay(tag, ep, RELAY_PORT + i, REGION, inference_component=ic)
        for i, (tag, (ep, ic)) in enumerate(ENDPOINTS.items())}

# Cached comparison tags have no live endpoint/relay — add them to URLS with a None placeholder. They're
# served from the archive (restore cell below), so run_rollouts() skips them and the None is never used.
for tag in CACHED_REF_TAGS:
    URLS[tag] = None

# Smoke test: confirm each live relay answers and that the thinking switch reaches the model. Cached tags skip.
from shopbench_harness import has_reasoning


def ask(url, enable_thinking=None):
    """One short completion through a relay. enable_thinking=None skips the switch (used for cached tags)."""
    client = OpenAI(base_url=url, api_key="unused")
    kw = {"extra_body": {"chat_template_kwargs": {"enable_thinking": enable_thinking}}} if enable_thinking is not None else {}
    return client.chat.completions.create(
        model="", max_tokens=1024, messages=[{"role": "user", "content": "What is 17*3?"}], **kw).choices[0]


def reasoned(choice):
    """True if the reply reasoned — a real <think> block, or it ran out of tokens mid-thought."""
    text = choice.message.content
    return has_reasoning(text) or (choice.finish_reason == "length" and "</think>" not in text)


for tag, url in URLS.items():
    if url is None:
        print(f"{tag:18s} cached (served from reference_evaluations/, no live call)")
        continue
    for flag in (True, False):
        choice = ask(url, enable_thinking=flag)
        print(f"{tag:18s} enable_thinking={flag!s:5s} reasoned: {reasoned(choice)!s:5s}  {choice.message.content[-40:]!r}")
# Expected: reasoned True with enable_thinking=True and False with False, for base and student.

## 5. Run the agents

All models run every held-out problem through the unmodified ShoppingBench agent loop, in parallel. Interrupting stops
the runs; re-running resumes (finished problems in `OUT` are kept, so use a new `RUN_NAME` for a new model).

### What the agent does on each problem

**Tools** (all run against the local catalog from step 1, so every model sees the same results):

<table align="left">
  <tr><th align="left"><b>Tool</b></th><th align="left"><b>Parameters</b></th><th align="left"><b>Returns</b></th></tr>
  <tr><td align="left"><code>find_product</code></td><td align="left"><code>q</code>: search text<br><code>page</code>: 1–5<br><code>price</code>: "low-high", e.g. "100-500"<br><code>service</code>: official, freeShipping, COD, flashsale<br><code>sort</code>: priceasc, pricedesc, order (most sold)<br><code>shop_id</code>: one shop only</td><td align="left">Up to 10 products: id, shop, title, price, services, sold count</td></tr>
  <tr><td align="left"><code>view_product_information</code></td><td align="left"><code>product_ids</code>: IDs joined by commas, e.g. "4410867766,4422385859"</td><td align="left">Full details: attributes and SKU options (size, color, ...)</td></tr>
  <tr><td align="left"><code>recommend_product</code></td><td align="left"><code>product_ids</code>: IDs joined by commas</td><td align="left">The agent's answer</td></tr>
  <tr><td align="left"><code>terminate</code></td><td align="left"><code>status</code>: e.g. success</td><td align="left">Ends the trajectory</td></tr>
</table>
<br clear="all">

**The loop (one trajectory).** At every step the model gets a system prompt with the tool descriptions plus the dialogue so far, and replies in this format:

```text
<think> reasoning ... </think>          <- only when thinking is on
<tool_call>[{"name": "find_product", "parameters": {"q": "baggy high waist denim jeans", "page": 1}}]</tool_call>
```

The harness parses the `<tool_call>` block, runs the tools, appends their results to the dialogue, and asks the model again. A trajectory ends when the agent calls `terminate`, after 30 steps, or when a reply cannot be parsed.

In [ ]:
import shutil

# Copy cached comparison rollouts (CACHED_REF_TAGS) from reference_evaluations/ into OUT, so run_rollouts()
# (next cell) sees them as already complete and skips them — no live calls for these tags.
REF_ARCHIVE = OUT / "reference_evaluations"

restored = []
for tag in CACHED_REF_TAGS:
    for t in EVAL_TASKS:
        for prefix in ("rollout", "cfg"):
            ext = "jsonl" if prefix == "rollout" else "json"
            src = REF_ARCHIVE / f"{prefix}_{tag}_{t}.{ext}"
            dst = OUT / f"{prefix}_{tag}_{t}.{ext}"
            if src.exists() and not dst.exists():
                shutil.copy2(src, dst)
                restored.append(dst.name)
            elif not src.exists():
                print(f"⚠️  missing cached file for {tag!r}: {src} — this tag will run live instead "
                      f"of being skipped (if that's not possible, this task's job will fail).")

if restored:
    print(f"Restored {len(restored)} cached reference/frontier file(s) from {REF_ARCHIVE}:")
    for name in restored:
        print(f"  {name}")
else:
    print(f"No files restored (either {REF_ARCHIVE} has nothing new, or OUT already has them).")

In [ ]:
JOBS = {}
for tag, url in URLS.items():
    for t in EVAL_TASKS:
        cfg, rollout = harness.write_rollout_cfg(OUT, tag, t, CLEAN[t], url, ENABLE_THINKING, threads=ROLLOUT_THREADS)
        JOBS[f"{tag}_{t}"] = (cfg, rollout, nlines(CLEAN[t]))
harness.run_rollouts(JOBS, poll_s=30)

## 6. Score the agents (ASR)

Each finished trajectory is scored against its problem's stored constraints (see below), per task and per model. The next cell computes, for every `(task, model)`:

- **`asr`** — Absolute Success Rate: the share of problems *solved* (the headline metric).
- **`exact_id`** / **`recommend`** — share where the exact gold product was recommended / where any real product was recommended at all.
- **health fields** (`empty_content`, `null_tool_results`, `no_terminate`, `steps_with_reasoning`) — diagnostics that tell a *broken run* (endpoint/search errors, unparsable replies) apart from a model that genuinely scored low. Always sanity-check these before trusting the ASR numbers.

It then prints the base→student ASR delta per task (and the gap closed toward the reference, if run), plus a count-averaged aggregate across intents, and returns a DataFrame with every metric.

**How a trajectory is scored.** Each test problem stores the constraints its gold product satisfies (price bounds, required services, SKU options/attributes, and a gold title). The recommended product does **not** have to be the gold one — it's checked against every stored constraint: title similarity to the gold ≥ 0.5 (`Qwen3-Embedding-0.6B`), all price bounds hold, every required service is offered, and every required (key, value) attribute/SKU pair is present. Recommending the exact gold product passes directly. A problem is **solved** per the "Solved when" column in the intro (product: the one product meets everything; shop: all from one shop; voucher: within budget after the voucher). These are ShoppingBench's own rules (`src/agent/rewards/orm.py`); `scripts/score_rollouts.py` calls the same functions. The metrics (`asr`, `exact_id`, `recommend`, plus health fields) are explained under "Reading the results" below.

In [ ]:
from eval_utils import score_all

# score every (model, task), isolate per-cell failures, build the table.
RESULTS, df, SCORE_ERRORS = score_all(harness, URLS, EVAL_TASKS, JOBS, CLEAN)

# Comparison tags, in display order (reference_qwen3-32b before frontier_medium); any other
# reference*/frontier* tag is appended by the sorted catch-all.
REF_TAGS = [t for t in ["reference_qwen3-32b", "frontier_medium"]
            if any((task, t) in RESULTS for task in EVAL_TASKS)]
REF_TAGS += sorted(t for (_, t) in RESULTS
                   if t.startswith(("reference", "frontier")) and t not in REF_TAGS)

# ---- base -> student (-> each comparison model) per intent ----
for t in EVAL_TASKS:
    b = RESULTS.get((t, "base"), {}).get("asr")
    s = RESULTS.get((t, "student"), {}).get("asr")
    if b is None or s is None:
        continue
    line = f"{t}: ASR base {b:.3f} -> student {s:.3f} ({100 * (s - b):+.1f} pts)"
    for rtag in REF_TAGS:
        ref = RESULTS.get((t, rtag), {}).get("asr")
        if ref is not None:
            gap = f"; student closes {(s - b) / (ref - b):.0%} of the gap" if ref > b else ""
            line += f"; {rtag} {ref:.3f}{gap}"
    print(line)

# ---- Aggregate = simple mean over intents (each intent counts equally); only full-coverage models ----
avg = {}
for tag in URLS:
    asrs = [RESULTS.get((t, tag), {}).get("asr") for t in EVAL_TASKS]
    if all(a is not None for a in asrs):
        avg[tag] = sum(asrs) / len(asrs)
if len(EVAL_TASKS) > 1 and avg:
    print(f"Avg. ASR ({', '.join(EVAL_TASKS)}; held-out, mean over intents): "
          + ", ".join(f"{tag} {a:.3f}" for tag, a in avg.items()))

if SCORE_ERRORS:
    print(f"\n⚠️  {len(SCORE_ERRORS)} of {len(URLS) * len(EVAL_TASKS)} (model, task) cells did not score and are "
          f"missing from the table: {sorted(SCORE_ERRORS)} — inspect SCORE_ERRORS for the reasons.")
df

## 7. Visualize the results

Two grouped bar charts, one bar per model (base / student / reference) across the three intents plus their average:

- **ASR** — the share of problems solved. The headline comparison: how far the distilled student moves above base, and how much of the gap to the frontier reference it closes.
- **Recommend rate** — the share of trajectories that recommended any real product. Reading the two panels together is diagnostic: if ASR rose but the recommend rate didn't, the student is picking *better* products; if the recommend rate itself rose, distillation also taught it to reliably finish with a recommendation (fewer unparsable / non-terminating trajectories).

The title notes whether thinking was on or off for the run.

In [ ]:
import importlib
import eval_utils
from eval_utils import plot_results

# ASR (did it solve the problem?) + recommend-rate (did it even finish with a recommendation?). Reusing `avg`
# keeps the chart's Avg. column identical to the printed aggregate.
plot_results(
    RESULTS, EVAL_TASKS,
    metrics=[("asr", "ASR (share of problems solved)"),
             ("recommend", "Share of trajectories that recommended a product")],
    avg=avg, enable_thinking=ENABLE_THINKING,
    labels={"base": "base", "student": "student"})

### Reading the results

- Compare base and student on `asr` (`n` = problems scored); `exact_id` = exact gold product; `recommend` = recommended
  any real product.
- Health (share of trajectories): `empty_content` and `null_tool_results` should be ~0 (endpoint or search errors);
  `no_terminate` = ended without finishing, mostly an unparsable reply (distillation lowers it);
  `steps_with_reasoning` ≈ 1 with thinking on, ≈ 0 off.

![ASR by task: base, student, reference_qwen3-32b, frontier_medium](asr_results_chart.png)

Reference run (LoRA r64, held-out, thinking off). Left: ASR by task (**bold** = `frontier_medium`, the strongest model here). Right: student gap vs. base, vs. the same-family dense reference, and vs. `frontier_medium`.

<div style="display:flex; gap:40px; align-items:flex-start; flex-wrap:wrap;">

<table>
  <tr><th align="left"><b>Model</b></th><th><b>product</b></th><th><b>shop</b></th><th><b>voucher</b></th><th><b>Avg.</b></th></tr>
  <tr><td align="left">base</td><td>0.15</td><td>0.06</td><td>0.04</td><td>0.09</td></tr>
  <tr><td align="left">student</td><td>0.48</td><td>0.36</td><td>0.20</td><td>0.35</td></tr>
  <tr><td align="left">reference_qwen3-32b</td><td>0.42</td><td>0.10</td><td>0.10</td><td>0.21</td></tr>
  <tr><td align="left">frontier_medium</td><td><b>0.57</b></td><td><b>0.66</b></td><td><b>0.34</b></td><td><b>0.52</b></td></tr>
</table>

<table>
  <tr><th align="left"><b>Δ student (points)</b></th><th><b>product</b></th><th><b>shop</b></th><th><b>voucher</b></th><th><b>Avg.</b></th></tr>
  <tr><td align="left">vs. base</td><td>+33</td><td>+30</td><td>+16</td><td>+26</td></tr>
  <tr><td align="left">vs. reference_qwen3-32b</td><td>+6</td><td>+26</td><td>+10</td><td>+14</td></tr>
  <tr><td align="left">vs. frontier_medium</td><td>−9</td><td>−30</td><td>−14</td><td>−17</td></tr>
</table>

</div>

The student closes most of the gap to the frontier comparison model while staying far cheaper to serve, and
clears the same-family dense reference model (`reference_qwen3-32b`) on every task. Shop gains the most from
distillation. Expect a few points of run-to-run variation.

## 8. IFEval (optional)

ShoppingBench measures the skill we distilled *for*; [IFEval](https://arxiv.org/abs/2311.07911) is the counterweight — it checks whether the student kept its **general instruction-following** ability or overfit to the shopping task. It scores verifiable format instructions ("answer in exactly 3 bullet points", "don't use the word 'the'"), so `prompt_strict` = the share of prompts where *every* rule was followed.

Read it as a **regression check**: a small drop vs. base is the expected, acceptable cost of specialization; a large drop means distillation narrowed the model too much. It runs with thinking **off**, and `reasoning_leaks` must be `0` for the score to be valid — if reasoning leaks into the answer, the format checker sees the wrong text and the number is meaningless.

In [ ]:
if RUN_IFEVAL:
    LMEVAL_ENV = WORK / "lmevalenv"
    if not (LMEVAL_ENV / "bin/lm_eval").exists():
        run(sys.executable, "-m", "venv", LMEVAL_ENV)
        run(LMEVAL_ENV / "bin/pip", "install", "-q", "lm_eval[api,ifeval]")
    IFEVAL = {}
    for i, (tag, (ep, ic)) in enumerate(ENDPOINTS.items()):
        url = relay.start_relay(f"ifeval_{tag}", ep, RELAY_PORT + 100 + i, REGION, inference_component=ic,
                                force={"chat_template_kwargs": {"enable_thinking": False}})
        out = subprocess.run([sys.executable, SCRIPTS / "run_ifeval.py", "--base-url", url, "--out", OUT / f"ifeval_{tag}",
                              "--lm-eval", LMEVAL_ENV / "bin/lm_eval"], capture_output=True, text=True)
        if out.returncode: raise RuntimeError(out.stderr[-3000:])
        IFEVAL[tag] = json.loads(out.stdout.strip().splitlines()[-1])
        print(tag, IFEVAL[tag], "" if IFEVAL[tag]["reasoning_leaks"] == 0 else "<- INVALID: reasoning leaked into predictions")

## 9. Delete the endpoints

A real-time endpoint **bills for its instance the whole time it's InService**, whether or not it's handling requests — so tear it down once you have your numbers. Run this cell when you're done (or before stepping away).

`teardown_endpoint` deletes the whole stack in dependency order — the **adapter inference component(s) first, then the base inference component, then the endpoint and its config** — because an adapter IC references its base and ICs don't cascade-delete. It finds the adapters structurally (by their `BaseInferenceComponentName`), so it cleans up correctly regardless of the exact component names. This also deletes a **reused** endpoint, so skip this cell if you want to keep the endpoint alive for another run (and record its names in `REUSE_ENDPOINTS`). It then stops the local search server and relays.

In [ ]:
# Tear down the endpoint and all its inference components (adapters first, then base, then the
# endpoint + config). teardown_endpoint finds adapters structurally (by BaseInferenceComponentName),
# so it cleans up the student IC + base IC + endpoint regardless of their exact names.
DO_CLEANUP = True

if DO_CLEANUP:
    kd_serve.teardown_endpoint(ENDPOINTS["base"][0], region=REGION)
    harness.stop_all()
    relay.stop_all()